In [1]:
%reload_kedro

                    INFO     Resolved project path as:                                              ]8;id=137147;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=489931;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#175\175]8;;\
                             /Users/nicolasbetancourt/Documents/GitHub/serpientes-de-colombia.                     
                             To set a different path, run '%reload_kedro <project_root>'                           

file
data/02_intermediate/train_labels_and_paths.csv
<class 'str'>
file
data/02_intermediate/validation_labels_and_paths.csv
<class 'str'>
file
data/02_intermediate/test_labels_and_paths.csv
<class 'str'>


                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=101591;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=937781;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro_telemetry/plugin.py#273\273]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project serpientes_de_colombia                                   ]8;id=363801;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=760865;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#141\141]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=766849;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=484760;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#142\142]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=59551;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=892508;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#148\148]8;;\

In [2]:
train_image_dataset = catalog.load("train_set")

                    INFO     Loading data from train_set (KedroPytorchImageDataset)...          ]8;id=76860;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=5632;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

In [3]:
import numpy as np

import torchvision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import  transforms
from PIL import Image
from torchvision.models import get_model
from torch.utils.data import WeightedRandomSampler
import time
import pandas as pd

In [4]:
train_image_dataset=catalog.load('train_set')
validation_image_dataset= catalog.load('validation_set')

                    INFO     Loading data from train_set (KedroPytorchImageDataset)...          ]8;id=464047;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=392836;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from validation_set (KedroPytorchImageDataset)...     ]8;id=831348;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=808388;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

In [5]:

training_params= catalog.load('params:training_params')
label_map= catalog.load('label_encoder')
device=catalog.load('device')
backbone_model=catalog.load("params:backbone_model")



                    INFO     Loading data from params:training_params (MemoryDataset)...        ]8;id=814916;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=827486;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from label_encoder (JSONDataset)...                   ]8;id=548863;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=335320;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from device (TextDataset)...                          ]8;id=25085;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=733944;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:backbone_model (MemoryDataset)...         ]8;id=313382;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=437321;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

# Sampling

In [6]:
train_image_dataset.data.label.value_counts()


label
falsa_vibora    1213
coral           1208
vibora          1206
falsa_coral     1187
Name: count, dtype: int64

In [7]:
sampling_size = 64

train_image_dataset.data = (
    train_image_dataset.data
    .sample(n=sampling_size, random_state=42)
    .reset_index(drop=True)
)

validation_image_dataset.data = (
    validation_image_dataset.data
    .sample(n=sampling_size, random_state=42)
    .reset_index(drop=True)
)

# Model pipeline

In [8]:
train_transform = transforms.Compose([
transforms.Resize((224, 224)),   # resize WITHOUT cropping
transforms.RandomHorizontalFlip(p=0.5),
transforms.RandomRotation(15),   # small rotations
transforms.ToTensor(),
transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
  ])

val_transform = transforms.Compose([
transforms.Resize((224, 224)),   # same resizing, no randomness
transforms.ToTensor(),
transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])
target_transform=lambda x: label_map.get(x)

training_set = train_image_dataset.with_transforms(
    transform=train_transform, 
    target_transform=target_transform
    )
validation_set = validation_image_dataset.with_transforms(
    transform=val_transform, 
    target_transform=target_transform
    )

In [9]:
training_generator = DataLoader(
    training_set,
    batch_size=training_params['generator_params']["batch_size"],
    #sampler=sampler,
    num_workers=training_params['generator_params'].get("num_workers", 0)
)
validation_generator = DataLoader(
    validation_set, 
**training_params['generator_params']
)
device = torch.device("cuda" if device=="cuda" and  torch.cuda.is_available() else "cpu")

In [10]:
model_conv = get_model(backbone_model, weights='DEFAULT')

In [11]:
num_ftrs = model_conv.fc.in_features
num_classes=train_image_dataset.data[train_image_dataset.label_column].nunique()
model_conv.fc = nn.Sequential(
                                nn.Linear(num_ftrs, 256),
                                nn.ReLU(),
                                nn.Dropout(0.3),
                                nn.Linear(256, num_classes)
                            )
for param in model_conv.parameters():
    param.requires_grad = True


model_conv = model_conv.to(device)


criterion = nn.CrossEntropyLoss()


optimizer_conv = torch.optim.SGD(model_conv.parameters(), lr=1e-4, momentum=0.9)

#exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

# model, train_dataloader, validation_dataloader, criterion, optimizer
#model_conv,loss_history=model_calibration(
#    model_conv, 
#    training_generator, 
#    validation_generator, 
#    criterion, 
#    optimizer_conv,
#    device, 
#    num_epochs=training_params["num_epochs"]
#    )

# model_calibration

In [12]:
model= model_conv
train_dataloader= training_generator
validation_dataloader= validation_generator
criterion= criterion
optimizer= optimizer_conv
device= device
num_epochs=300

In [ ]:
model.to(device)
loss_history = []
for epoch in range(num_epochs):
    model.eval()
    model.fc.train()
    #model.train()
    
    train_loss = 0.0
    for idx, (x, y, _) in enumerate(train_dataloader):   # <- ignore path with _
        x = x.to(device)
        y = y.to(device)
    
        outputs = model(x)
        loss = criterion(outputs, y)
        loss_history.append({'loss':loss.item(), 'batch': idx, 'epoch': epoch, 'set': 'train'})
    
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        #train_loss += loss.item()
    
    #train_loss /= len(train_dataloader)
    
    # ---- Validation ----
    val_loss = 0.0
    model.eval()
    with torch.no_grad():
        correct=0
        total=0
        for idx, (x, y, _) in enumerate(validation_dataloader):  
            x = x.to(device)
            y = y.to(device)
    
            outputs = model(x)
            loss = criterion(outputs, y)
            loss_history.append({'loss':loss.item(), 'batch': idx, 'epoch': epoch, 'set': 'validation'})
            
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
            #val_loss += loss.item()
    
    #val_loss /= len(validation_dataloader)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    
                    
    
       
   

Epoch 1/300
Epoch 2/300
Epoch 3/300
Epoch 4/300
Epoch 5/300
Epoch 6/300
Epoch 7/300
Epoch 8/300
Epoch 9/300
Epoch 10/300
Epoch 11/300
Epoch 12/300
Epoch 13/300
Epoch 14/300
Epoch 15/300
Epoch 16/300
Epoch 17/300
Epoch 18/300
Epoch 19/300
Epoch 20/300
Epoch 21/300
Epoch 22/300
Epoch 23/300
Epoch 24/300
Epoch 25/300
Epoch 26/300
Epoch 27/300
Epoch 28/300
Epoch 29/300
Epoch 30/300
Epoch 31/300
Epoch 32/300
Epoch 33/300
Epoch 34/300
Epoch 35/300
Epoch 36/300
Epoch 37/300
Epoch 38/300
Epoch 39/300
Epoch 40/300
Epoch 41/300
Epoch 42/300
Epoch 43/300
Epoch 44/300
Epoch 45/300
Epoch 46/300
Epoch 47/300
Epoch 48/300
Epoch 49/300
Epoch 50/300
Epoch 51/300
Epoch 52/300
Epoch 53/300
Epoch 54/300
Epoch 55/300
Epoch 56/300
Epoch 57/300
Epoch 58/300
Epoch 59/300
Epoch 60/300
Epoch 61/300
Epoch 62/300
Epoch 63/300
Epoch 64/300
Epoch 65/300
Epoch 66/300
Epoch 67/300
Epoch 68/300
Epoch 69/300
Epoch 70/300
Epoch 71/300
Epoch 72/300
Epoch 73/300
Epoch 74/300
Epoch 75/300
Epoch 76/300
Epoch 77/300
Epoch 78

In [ ]:
loss_history=pd.DataFrame(loss_history)

# Model output

In [ ]:
test_set=catalog.load('test_set')

In [ ]:
def get_predictions(data,model, label_map, training_params, device):
    transform=transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    target_transform=lambda x: label_map.get(x)
    test_set = data.with_transforms(transform=transform, )

    test_generator = DataLoader(test_set, **training_params['generator_params'])

    model.eval()
    predictions = []
    labels = []
    paths = []
    inverse_label_map={v:k for k,v in label_map.items()}
    model=model.to(device)
    with torch.no_grad():
        for X, y, path in test_generator:
            X = X.to(device)
            preds = model(X).argmax(1).cpu()  # get predicted classes
            
            predictions.extend([ inverse_label_map[k.item()] for k in  preds])
            labels.extend(y)
            paths.extend(path)
    test_data_with_predictions=pd.DataFrame({'path':paths,
              'label': labels,
              'prediction': predictions
             })
    return test_data_with_predictions

In [ ]:
predictions=get_predictions(
test_set,
    model, 
label_map, 
training_params, 
device)

In [ ]:
predictions_train=get_predictions(
    training_set,
    model, 
label_map, 
training_params, 
device)

# Reporting

In [ ]:
def cf_matrix(predictions, label_encoder):
    label_order = list(label_encoder.keys())

    cm = confusion_matrix(
        predictions['label'],
        predictions['prediction'],
        labels=label_order
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_order
    )

    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title("Confusion Matrix")

    #return fig

def loss_history_plot(loss_history):
    loss_history=loss_history.groupby(['set','epoch']).agg(loss=('loss','mean')).reset_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    for set_type in ['train', 'validation']:
        subset = loss_history[loss_history['set'] == set_type]
        ax.plot(subset['epoch'], subset['loss'], label=set_type)
    ax.set_title(f'Loss History')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    #return fig

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
cf_matrix(predictions_train, label_map)

In [ ]:
loss_history_plot(loss_history)